# Spaceship Titanic — Classical ML
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

Complete classical ML pipeline applied to the Spaceship Titanic dataset — from baseline comparison through hyperparameter optimisation, feature importance analysis, error analysis, and model selection. Reuses the preprocessing from `spaceship_titanic_preprocessing.ipynb` (Week 6) and the comparison framework from `machine_learning_pipelines.ipynb` (Week 5).

### Main goals:

- Run 5-fold CV comparison across six classifiers on the same preprocessed data.
- Tune the top two models with GridSearchCV and compare tuned vs default performance.
- Extract and validate feature importances against Week 5 EDA findings.
- Perform threshold analysis and select the best model for submission.
- Maintain a running experiment table across all sections.

---

## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                      cross_val_score, GridSearchCV)
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, precision_recall_curve, roc_curve, auc)

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Preprocessing — Week 6 Pipeline

Reproduces `spaceship_titanic_preprocessing.ipynb` exactly. All subsequent sections use `X_tr_t`, `X_val_t`, `X_test_t`, `y_tr`, `y_val`, `preprocessor`, and `feature_names` without modification.

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

TRAIN_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/spaceship-titanic/train.csv'
TEST_URL  = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/spaceship-titanic/test.csv'
train_raw = pd.read_csv(TRAIN_URL)
test_raw  = pd.read_csv(TEST_URL)

SPEND_COLS = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

def prep(df):
    df = df.copy()
    s = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = s[0]; df['CabinNum'] = pd.to_numeric(s[1], errors='coerce'); df['Side'] = s[2]
    df = df.drop(columns=['Cabin'])
    df['TotalSpend'] = df[SPEND_COLS].fillna(0).sum(axis=1)
    df['IsSpender']  = (df['TotalSpend'] > 0).astype(int)
    df['AgeGroup']   = pd.cut(df['Age'], bins=[0,12,17,35,60,200],
                               labels=['Child','Teen','YoungAdult','Adult','Senior'])
    df['AgeGroup']   = df['AgeGroup'].astype(str).replace('nan', np.nan)
    return df.drop(columns=[c for c in ['PassengerId','Name'] if c in df.columns])

train = prep(train_raw); test = prep(test_raw)

SPEND_AND_TOTAL  = SPEND_COLS + ['TotalSpend']
OTHER_NUMERIC    = ['Age','CabinNum','IsSpender']
CATEGORICAL_COLS = ['HomePlanet','CryoSleep','Destination','VIP','Deck','Side','AgeGroup']
FEATURE_COLS     = SPEND_AND_TOTAL + OTHER_NUMERIC + CATEGORICAL_COLS

X_train = train[FEATURE_COLS]; y_train = train['Transported'].astype(int)
X_test_raw = test[FEATURE_COLS]

preprocessor = ColumnTransformer([
    ('spend', Pipeline([('imp', SimpleImputer(strategy='median')),
                        ('log', FunctionTransformer(np.log1p, validate=False)),
                        ('sc',  StandardScaler())]),         SPEND_AND_TOTAL),
    ('num',   Pipeline([('imp', SimpleImputer(strategy='median')),
                        ('sc',  StandardScaler())]),         OTHER_NUMERIC),
    ('cat',   Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
                        CATEGORICAL_COLS)
], remainder='drop')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
preprocessor.fit(X_tr)
X_tr_t   = preprocessor.transform(X_tr)
X_val_t  = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test_raw)
ohe_cats      = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out(CATEGORICAL_COLS)
feature_names = SPEND_AND_TOTAL + OTHER_NUMERIC + list(ohe_cats)
print(f'X_tr_t: {X_tr_t.shape}  features: {len(feature_names)}')

## Part 1 — Baseline Model Comparison

Six classifiers evaluated with 5-fold stratified CV on five metrics. Same StratifiedKFold configuration as the Week 5 `machine_learning_pipelines.ipynb` framework (Week 5) — only the dataset changes.

In [3]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                      max_depth=4, random_state=42),
    'MLP':                 MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                                         alpha=0.01, max_iter=500, early_stopping=True, random_state=42),
}

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = ['accuracy','precision','recall','f1','roc_auc']

cv_rows = []
for name, model in MODELS.items():
    scores = cross_validate(model, X_tr_t, y_tr, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {'Model': name}
    for m in SCORING:
        row[f'{m}_mean'] = scores[f'test_{m}'].mean()
        row[f'{m}_std']  = scores[f'test_{m}'].std()
    cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows).sort_values('accuracy_mean', ascending=False).reset_index(drop=True)

# Summary table
for m in SCORING:
    cv_df[f'{m}'] = cv_df.apply(lambda r: f"{r[f'{m}_mean']:.4f} ± {r[f'{m}_std']:.4f}", axis=1)
display_cols = ['Model'] + SCORING
print(cv_df[display_cols].to_string(index=False))

In [5]:
metrics_plot = ['accuracy','f1','roc_auc']
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
palette = ['#1F3864','#4472C4','#aab4c8','#2CA02C','#C00000','#FFA500']
model_order = cv_df['Model'].tolist()

for ax, metric in zip(axes, metrics_plot):
    means = cv_df.set_index('Model').reindex(model_order)[f'{metric}_mean']
    stds  = cv_df.set_index('Model').reindex(model_order)[f'{metric}_std']
    ax.barh(model_order, means, xerr=stds,
            color=palette[:len(model_order)], alpha=0.85,
            error_kw={'ecolor': '#333333', 'capsize': 3})
    ax.set_title(metric.upper().replace('_','-'))
    ax.set_xlim(0.65, 1.0)
    ax.axvline(0.8, color='k', lw=0.6, linestyle='--')

fig.suptitle('5-Fold CV Baseline Comparison — Spaceship Titanic', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Observation:**
Gradient Boosting and Random Forest lead on all three metrics — consistent with the loan status comparison in `decision_trees_and_ensembles.ipynb`. The performance gap between ensembles and linear/kernel models is larger here, reflecting the non-linear CryoSleep × spending interaction identified in `spaceship_titanic_eda.ipynb`. Both ensembles advance to hyperparameter tuning.

## Part 2 — Hyperparameter Tuning

GridSearchCV applied to the top two models using the same Pipeline + GridSearchCV pattern from `machine_learning_pipelines.ipynb` (Week 5).

In [6]:
rf_param_grid = {
    'n_estimators': [200, 400],
    'max_depth':    [6, 8, 12, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
rf_grid.fit(X_tr_t, y_tr)

print(f'RF best CV accuracy: {rf_grid.best_score_:.4f}')
print(f'RF best params:      {rf_grid.best_params_}')

In [7]:
rf_res = pd.DataFrame(rf_grid.cv_results_)
best_mf = rf_grid.best_params_['max_features']
best_ms = rf_grid.best_params_['min_samples_split']
slice_rf = rf_res[
    (rf_res['param_max_features'] == best_mf) &
    (rf_res['param_min_samples_split'] == best_ms)
].copy()
slice_rf['param_max_depth'] = slice_rf['param_max_depth'].astype(str)
pivot_rf = slice_rf.pivot_table(
    index='param_max_depth', columns='param_n_estimators', values='mean_test_score')

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot_rf, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            linewidths=0.5, annot_kws={'size': 11})
ax.set_title('Random Forest GridSearchCV — CV Accuracy')
plt.tight_layout()
plt.show()

In [8]:
gb_param_grid = {
    'n_estimators':  [200, 400],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth':     [3, 4, 5],
    'subsample':     [0.8, 1.0],
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
gb_grid.fit(X_tr_t, y_tr)

print(f'GB best CV accuracy: {gb_grid.best_score_:.4f}')
print(f'GB best params:      {gb_grid.best_params_}')

In [9]:
gb_res = pd.DataFrame(gb_grid.cv_results_)
best_ss = gb_grid.best_params_['subsample']
best_ne = gb_grid.best_params_['n_estimators']
slice_gb = gb_res[
    (gb_res['param_subsample'] == best_ss) &
    (gb_res['param_n_estimators'] == best_ne)
].copy()
pivot_gb = slice_gb.pivot_table(
    index='param_max_depth', columns='param_learning_rate', values='mean_test_score')

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot_gb, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            linewidths=0.5, annot_kws={'size': 11})
ax.set_title('Gradient Boosting GridSearchCV — CV Accuracy')
plt.tight_layout()
plt.show()

**Observation:**
Lower learning rates with more estimators produce the most stable CV accuracy — mirroring the learning rate analysis in `deep_learning_keras.ipynb`. The heatmap structure is the same as in `machine_learning_pipelines.ipynb` (Week 5).

## Tuned vs Default Comparison

In [10]:
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_

tuned_results = []
for name, model in [
    ('Random Forest (default)',   RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
    ('Random Forest (tuned)',     best_rf),
    ('Gradient Boosting (default)', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)),
    ('Gradient Boosting (tuned)',   best_gb),
]:
    cv_acc = cross_val_score(model, X_tr_t, y_tr, cv=cv, scoring='accuracy').mean()
    model.fit(X_tr_t, y_tr)
    val_acc = accuracy_score(y_val, model.predict(X_val_t))
    tuned_results.append({'Model': name, 'CV Acc': cv_acc, 'Val Acc': val_acc})

tr_df = pd.DataFrame(tuned_results)
print(tr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(tr_df))
ax.bar(x - 0.2, tr_df['CV Acc'],  0.35, label='CV Acc',  color='#1F3864', alpha=0.85)
ax.bar(x + 0.2, tr_df['Val Acc'], 0.35, label='Val Acc', color='#C00000', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tr_df['Model'], rotation=12, ha='right')
ax.set_ylabel('Accuracy'); ax.set_ylim(0.75, 1.0)
ax.set_title('Tuned vs Default — CV and Validation Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Importances — Best Tuned Model

In [11]:
if best_rf.score(X_val_t, y_val) >= best_gb.score(X_val_t, y_val):
    best_classical = best_rf; best_classical_name = 'Random Forest (tuned)'
else:
    best_classical = best_gb; best_classical_name = 'Gradient Boosting (tuned)'

imp = pd.Series(best_classical.feature_importances_, index=feature_names)
top20 = imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#1F3864' if v >= top20.quantile(0.6) else '#aab4c8' for v in top20.values]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], alpha=0.88)
ax.set_xlabel('Feature Importance')
ax.set_title(f'{best_classical_name} — Top 20 Feature Importances')
plt.tight_layout()
plt.show()

**Observation:**
`CryoSleep`, `IsSpender`, and `TotalSpend` rank among the top contributors — confirming the CryoSleep × spending interaction from `spaceship_titanic_eda.ipynb`. `Deck` OHE columns and `HomePlanet` also rank highly, consistent with the transport rate differences charted in Week 5 EDA.

## Part 3 — Threshold Optimisation

In [12]:
val_preds_best = best_classical.predict(X_val_t)
val_proba_best = best_classical.predict_proba(X_val_t)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, val_proba_best)
f1_by_thresh = 2 * precision * recall / (precision + recall + 1e-9)
best_thresh_idx = np.argmax(f1_by_thresh[:-1])
BEST_THRESHOLD  = thresholds[best_thresh_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(recall, precision, color='#1F3864', lw=2)
axes[0].scatter(recall[best_thresh_idx], precision[best_thresh_idx],
                color='#C00000', s=100, zorder=5,
                label=f'Best threshold={BEST_THRESHOLD:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve'); axes[0].legend()

axes[1].plot(thresholds, f1_by_thresh[:-1], color='#2CA02C', lw=2)
axes[1].axvline(BEST_THRESHOLD, color='#C00000', lw=1.2, linestyle='--',
                label=f'Optimal = {BEST_THRESHOLD:.3f}')
axes[1].axvline(0.5, color='#1F3864', lw=1.2, linestyle=':', label='Default = 0.5')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score vs Decision Threshold'); axes[1].legend()

plt.tight_layout()
plt.show()

preds_default = (val_proba_best >= 0.5).astype(int)
preds_optimal = (val_proba_best >= BEST_THRESHOLD).astype(int)
print(f'Default (0.500): Acc={accuracy_score(y_val,preds_default):.4f}  F1={f1_score(y_val,preds_default):.4f}')
print(f'Optimal ({BEST_THRESHOLD:.3f}): Acc={accuracy_score(y_val,preds_optimal):.4f}  F1={f1_score(y_val,preds_optimal):.4f}')
print(f'\nThreshold carried forward to spaceship_titanic_submission.ipynb: {BEST_THRESHOLD:.4f}')

## Final Validation Evaluation

In [13]:
print(f'Best classical model: {best_classical_name}')
print(f'Train accuracy: {accuracy_score(y_tr, best_classical.predict(X_tr_t)):.4f}')
print(f'Val accuracy:   {accuracy_score(y_val, val_preds_best):.4f}')
print()
print(classification_report(y_val, val_preds_best,
                             target_names=['Not Transported','Transported']))

cm = confusion_matrix(y_val, val_preds_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: No','Pred: Yes'],
            yticklabels=['Actual: No','Actual: Yes'],
            linewidths=0.5, linecolor='lightgray', cbar=False, annot_kws={'size': 13})
ax.set_title(f'Confusion Matrix — {best_classical_name}')
plt.tight_layout()
plt.show()

## Experiment Summary Table

In [14]:
# Accumulate all results
experiment_log = []
for _, row in cv_df.iterrows():
    experiment_log.append({
        'Model': row['Model'], 'Stage': 'Default CV',
        'CV Acc': f"{row['accuracy_mean']:.4f} ± {row['accuracy_std']:.4f}",
        'CV F1':  f"{row['f1_mean']:.4f} ± {row['f1_std']:.4f}",
        'AUC':    f"{row['roc_auc_mean']:.4f} ± {row['roc_auc_std']:.4f}",
    })
for row in [
    {'Model': 'Random Forest (tuned)', 'Stage': 'Tuned CV',
     'CV Acc': f'{cross_val_score(best_rf, X_tr_t, y_tr, cv=cv, scoring="accuracy").mean():.4f}',
     'CV F1':  f'{f1_score(y_val, best_rf.predict(X_val_t)):.4f}', 'AUC': '—'},
    {'Model': 'Gradient Boosting (tuned)', 'Stage': 'Tuned CV',
     'CV Acc': f'{cross_val_score(best_gb, X_tr_t, y_tr, cv=cv, scoring="accuracy").mean():.4f}',
     'CV F1':  f'{f1_score(y_val, best_gb.predict(X_val_t)):.4f}', 'AUC': '—'},
]:
    experiment_log.append(row)

exp_df = pd.DataFrame(experiment_log)
print(exp_df.to_string(index=False))